In [ ]:
# load data fitur hasil ekstraksi dari preprocessing.ipynb

import os
import json
import time
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix,
                             ConfusionMatrixDisplay)

SEED = 42
os.makedirs("results", exist_ok=True)

data = np.load("data/features.npz")
X_train, y_train = data["X_train"], data["y_train"]
X_test,  y_test  = data["X_test"],  data["y_test"]
class_names = data["class_names"]

print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Jumlah kelas:", len(class_names))

In [ ]:
# scaling fitur

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print("Mean train setelah scaling:", X_train_s.mean().round(4))
print("Std train setelah scaling :", X_train_s.std().round(4))

In [ ]:
# training MLP

mlp = MLPClassifier(
    hidden_layer_sizes=(256, 128),
    activation="relu",
    solver="adam",
    alpha=1e-3,
    learning_rate_init=1e-3,
    batch_size=64,
    max_iter=200,
    early_stopping=False,
    random_state=SEED,
    verbose=True,
)

start = time.time()
mlp.fit(X_train_s, y_train)
train_time = time.time() - start

print(f"Waktu training : {train_time:.1f} detik")
print("Jumlah epoch   :", mlp.n_iter_)
print("Loss akhir     :", round(mlp.loss_, 4))

In [ ]:
# kurva loss training

plt.figure(figsize=(7, 4))
plt.plot(mlp.loss_curve_)
plt.title("Kurva Loss Training MLP")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("results/figures/mlp_loss_curve.png", dpi=150)
plt.show()

In [ ]:
# evaluasi model pada data testing

y_pred = mlp.predict(X_test_s)

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, average="macro", zero_division=0)
rec  = recall_score(y_test, y_pred, average="macro", zero_division=0)
f1   = f1_score(y_test, y_pred, average="macro", zero_division=0)

print(f"Accuracy  : {acc:.4f}")
print(f"Precision : {prec:.4f}")
print(f"Recall    : {rec:.4f}")
print(f"F1-score  : {f1:.4f}")

print("\nLaporan per kelas:")
print(classification_report(y_test, y_pred, target_names=class_names, zero_division=0))

# Akurasi train sebagai pembanding untuk melihat overfitting
train_acc = accuracy_score(y_train, mlp.predict(X_train_s))
print(f"Akurasi train: {train_acc:.4f} | Akurasi test: {acc:.4f}")

In [ ]:
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(10, 9))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(
    ax=ax, cmap="Blues", xticks_rotation=90, colorbar=False
)
ax.set_title("Confusion Matrix MLP")
plt.tight_layout()
plt.savefig("results/figures/mlp_confusion_matrix.png", dpi=150)
plt.show()

In [ ]:
# simpan model MLP ke file

hasil = {
    "model": "MLP",
    "accuracy": float(acc),
    "precision_macro": float(prec),
    "recall_macro": float(rec),
    "f1_macro": float(f1),
    "train_accuracy": float(train_acc),
    "train_time_sec": float(train_time),
    "params": {
        "hidden_layer_sizes": [256, 128],
        "activation": "relu",
        "solver": "adam",
        "alpha": 1e-3,
        "max_iter": 200,
        "batch_size": 64,
    },
}

with open("results/mlp_metrics.json", "w") as f:
    json.dump(hasil, f, indent=2)

np.save("results/mlp_y_pred.npy", y_pred)
print("Tersimpan di results/mlp_metrics.json")